[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day34-gguf-llama-cpp-k-quants.ipynb)

# Day 34 — GGUF / llama.cpp K-Quants

**Tags:** `CPU-OK` · **Time:** 30–60 min · **Prereqs:** Day 29 (affine quantization, granularity), Day 30 (GPTQ), Day 33 (FP8 vs INT4)

GGUF is the ZIP file of local inference: one memory-mappable container holding metadata, tokenizer, and weights. K-quants fill that container with a superblock layout (256 weights, 8 sub-blocks of 32, 6-bit scale+min each) plus a service tier for tensors: S spends 4-bit everywhere, M upgrades half of `attn.wv`/`ffn.w2` to 6-bit. Today you run a real Q4_K_M model on a CPU, rederive the bit budgets in Python, and project the 16 GB laptop context budget.

> Colab note: we use a 0.5B Qwen Q4_K_M (~400 MB) to respect Colab disk. The same commands work for the 8B file on your own machine (Step 7 in the packet).


In [ ]:
# Cell 1: one-time installs. CPU-only day; llama.cpp shines on CPU.
# T4 optional but pointless here — the point is the CPU path.
%pip install --quiet llama-cpp-python huggingface_hub
print("llama-cpp-python installed; running on CPU")
# Expected: pip output suppressed, then the confirmation line.


## Part 1 — Download a Q4_K_M model

`bartowski` publishes GGUF quants of popular models. Qwen2.5-0.5B-Instruct Q4_K_M is ~400 MB — small enough for Colab, and the same inspection/generation code runs unchanged on the 8B file.


In [ ]:
# Cell 2: download the GGUF (~400 MB)
from huggingface_hub import hf_hub_download
import os

gguf_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-0.5B-Instruct-GGUF",
    filename="Qwen2.5-0.5B-Instruct-Q4_K_M.gguf",
)
size_mb = os.path.getsize(gguf_path) / 1024**2
print(f"file: {os.path.basename(gguf_path)}")
print(f"size: {size_mb:.1f} MB")
# Expected: Qwen2.5-0.5B-Instruct-Q4_K_M.gguf, size ~380-420 MB.
# Sanity check vs packet math: 0.49B params x ~4.75 bits / 8 ~ 290 MB
# + embeddings/output at Q6_K and metadata overhead -> ~400 MB. Close.


## Part 2 — Read the GGUF header

Magic `GGUF`, version, then a key-value metadata store and the tensor table. We parse just enough to count tensors and see which quant types each tensor uses — the S/M/L mix, visible in the wild.


In [ ]:
# Cell 3: minimal GGUF header reader (magic, version, counts, tensor types)
import struct
from collections import Counter

def read_gguf_header(path, max_tensors=12):
    with open(path, "rb") as f:
        magic = f.read(4)
        assert magic == b"GGUF", f"bad magic: {magic}"
        version, n_tensors, n_kv = struct.unpack("<IQQ", f.read(24))
        print(f"magic={magic} version={version} n_tensors={n_tensors} n_kv={n_kv}")
        # skip KV store; jump via tensor infos: each info = name_len, name,
        # n_dims, dims, dtype, offset. We walk them to tally dtypes.
        types = Counter()
        names = []
        for _ in range(n_tensors):
            (name_len,) = struct.unpack("<Q", f.read(8))
            name = f.read(name_len).decode()
            (n_dims,) = struct.unpack("<I", f.read(4))
            f.read(8 * n_dims)          # dims
            (dtype,) = struct.unpack("<I", f.read(4))
            f.read(8)                   # offset
            types[dtype] += 1
            if len(names) < max_tensors:
                names.append((name, dtype))
        return types, names

# GGUF type ids: 2=F16, 7=Q4_0, 10=Q8_0, 12=Q4_K, 13=Q5_K, 14=Q6_K, 24=Q2_K ...
TYPE_NAMES = {0:"F32",1:"F16",2:"F16",7:"Q4_0",8:"Q4_1",10:"Q8_0",12:"Q4_K",
              13:"Q5_K",14:"Q6_K",15:"Q2_K",16:"Q3_K",24:"IQ2_XXS"}
types, names = read_gguf_header(gguf_path)
print("tensor type tally:", {TYPE_NAMES.get(t, t): c for t, c in types.items()})
print("first tensors:")
for n, t in names:
    print(f"  {TYPE_NAMES.get(t, t):>8}  {n}")
# Expected: magic=b'GGUF', n_tensors in the hundreds, n_kv dozens.
# Tally shows mostly Q4_K with Q6_K for embeddings/output (the M-tier mix),
# F32/F16 for norms. If embeddings show Q6_K, you've seen the S/M/L mix live.


## Part 3 — Generate on CPU: prompt-eval vs generation tok/s

Decode is GEMV (weight stream per token, bandwidth-bound); prompt eval is batched GEMM (weights stream once for many tokens). Measure both, separately.


In [ ]:
# Cell 4: load + generate 100 tokens, split prompt-eval vs generation timing
import time
from llama_cpp import Llama

llm = Llama(model_path=gguf_path, n_ctx=2048, n_threads=4, verbose=False)
prompt = "Explain the KV cache in one paragraph."
print("warming up...", flush=True)
_ = llm(prompt, max_tokens=8, temperature=0.0)   # warm the kernels

t0 = time.perf_counter()
out = llm(prompt, max_tokens=100, temperature=0.0)
t1 = time.perf_counter()
u = out["usage"]
pe_toks, pe_s = u["prompt_tokens"], u["prompt_eval_time"] / 1e9 if "prompt_eval_time" in u else None
gen_toks = u["completion_tokens"]
gen_s = (t1 - t0)
print(f"prompt tokens: {pe_toks}")
print(f"generation: {gen_toks} tokens in {gen_s:.2f} s -> {gen_toks/gen_s:.1f} tok/s")
print()
print(out["choices"][0]["text"][:400])
# Expected: generation ~ tens of tok/s on Colab CPU (2 vCPU; slower than a
# laptop's 8 cores). Prompt eval is the fast part — check timing breakdown
# if your llama-cpp build reports prompt_eval_time. The GEMM-vs-GEMV gap
# from the packet should be visible: prompt eval several x faster.


## Part 4 — Rederive the K-quant bit budgets

Pure arithmetic, no model needed. Superblock = 256 weights; Q4_K/Q5_K use 8 sub-blocks of 32 with 6-bit scale + 6-bit min; Q2_K/Q3_K use 16 sub-blocks of 16 with 8-bit scale + 8-bit min; Q6_K uses an 8-bit scale per 16 weights; Q8_0 uses an fp16 scale per 32 weights.


In [ ]:
# Cell 5: bits/weight from the superblock layout — reproduce the packet table
def bits_per_weight(value_bits, n_sub, sub_size, scale_bits, min_bits):
    data = 256 * value_bits
    book = n_sub * (scale_bits + min_bits)
    return (data + book) / 256

table = {
    "Q2_K": bits_per_weight(2, 16, 16, 8, 8),
    "Q3_K": bits_per_weight(3, 16, 16, 8, 8),
    "Q4_K": bits_per_weight(4, 8, 32, 6, 6),
    "Q5_K": bits_per_weight(5, 8, 32, 6, 6),
    "Q6_K": (256*6 + 16*8) / 256,   # 8-bit scale per 16 weights, no min
    "Q8_0": (32*8 + 16) / 32,       # fp16 scale per 32 weights
}
for k, v in table.items():
    print(f"{k:>5}: {v:.3f} bits/weight")
# Expected:
#  Q2_K: 3.000 | Q3_K: 4.000 | Q4_K: 4.375
#  Q5_K: 5.375 | Q6_K: 6.250 | Q8_0: 8.500
# Exact match with the packet table — it is pure arithmetic.


## Part 5 — Scale-up: 8B file size and the 16 GB laptop context budget

Llama-3.1-8B: 8.03B params, 32 layers, GQA with 8 KV heads, head dim 128. Q4_K_M ≈ 20% of params at Q6_K (6.25 b/w), 80% at Q4_K (4.375 b/w).


In [ ]:
# Cell 6: project the 8B Q4_K_M size + laptop KV headroom
params = 8.03e9
eff_bits = 0.2 * 6.25 + 0.8 * 4.375
file_gb = params * eff_bits / 8 / 1024**3
print(f"effective bits/weight: {eff_bits:.2f}")
print(f"projected Q4_K_M file: {file_gb:.2f} GB  (published: ~4.9 GB)")

kv_per_token = 2 * 32 * 8 * 128 * 2   # K,V x layers x kv_heads x head_dim x fp16
print(f"KV per token: {kv_per_token/1024:.0f} KiB")
free_gb = 16 - 1 - file_gb            # 16 GB laptop, ~1 GB OS reserve
headroom = free_gb * 1024**3 / kv_per_token
print(f"free for KV: {free_gb:.1f} GB -> ~{headroom:,.0f} tokens of headroom")
# Expected:
#   effective bits/weight: 4.75
#   projected Q4_K_M file: ~4.77 GB (published ~4.9 GB — method validated)
#   KV per token: 128 KiB
#   free for KV: ~10.2 GB -> ~80,000+ tokens of headroom
# Compare with your Part 1 file: scale 0.49B -> 8.03B by the same ratio.


## Part 6 — The decision table (fill in your answers)

From the packet's Section 5. The H100 row is the deliberate trap — write down *why* GGUF is the wrong answer there.

| Hardware | Your pick | One-line why |
|---|---|---|
| 16 GB laptop (CPU) | | |
| T4 16 GB (Colab) | | |
| H100 80 GB | | |

Reference answers: laptop → **Q4_K_M** (4.9 GB weights + ~80k-token KV headroom; CPU kernels are GGUF-native). T4 → **Q8_0** or Q5_K_M (8.6 GB fits with KV room; ~zero perplexity loss). H100 → **fp16/FP8 under vLLM, not GGUF** (server kernels + PagedAttention beat llama.cpp on datacenter GPUs).

## Part 7 — Own machine (optional)

Repeat Part 3 with `Llama-3.1-8B-Instruct-Q4_K_M.gguf` (~4.9 GB, from `bartowski` or `lmstudio-community`) under `llama-cli`: `./llama-cli -m model.gguf -p "Explain KV cache in one paragraph" -n 100`. Expect ~15–25 tok/s generation on an 8-core laptop CPU — the ceiling from the packet, in your own terminal.
